# 05 — Canonical structure and provenance
**Goal:** Convert parser-specific output into one data model, with sections, page numbers, and access metadata.

**Must:** stable document ID, source URI/path, page or section provenance, access metadata.
**Trade-off:** storing only flat text is easier, but harms citations and section-aware chunking.

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
import hashlib
DATA = Path('../data') if Path('../data').exists() else Path('data')
source = DATA / 'jes2_manual.md'
text = source.read_text(encoding='utf-8')

@dataclass
class Element:
    kind: str
    text: str
    section: str
    source: str

elements = []
current_section = 'Document'
for line in text.splitlines():
    line = line.strip()
    if not line:
        continue
    if line.startswith('#'):
        current_section = line.lstrip('#').strip()
        elements.append(Element('heading', current_section, current_section, source.name))
    elif line.startswith('|'):
        elements.append(Element('table_line', line, current_section, source.name))
    else:
        elements.append(Element('paragraph', line, current_section, source.name))

canonical = {
    'document_id': hashlib.sha256(text.encode('utf-8')).hexdigest()[:16],
    'source': source.name,
    'allowed_groups': ['rag-demo-users'],  # synthetic ACL example
    'elements': [asdict(e) for e in elements]
}
print('Elements:', len(elements))
print(canonical['elements'][:4])

### Exercise
A production parser should type tables as structured rows/cells, not `table_line` strings. Enhance the Markdown table parsing into a `table` object.

**Expected:** each paragraph has a section path and source.

**Architecture choice:** keep the immutable original, a normalized representation, and search-specific text separate.